# DevForge Track 1: Can It Play DOOM? — Colab GPU Training Runner

**Team:** Prit (ML Lead), Surbhi (Curriculum), Vaibhav (Infra & Training), Dia (Evaluation)

### Overnight Run Schedule (IST):
- **22:00 Thu:** Launch Stage 1–5 Curriculum Run on Google Colab GPU (T4/A100)
- **00:00–02:30 Fri:** Active Watch: Vaibhav (check loss, step progress, Drive backups)
- **02:30–05:00 Fri:** Active Watch: Surbhi
- **05:00–07:30 Fri:** Active Watch: Dia
- **07:30–08:00 Fri:** Best Checkpoint Selection & Evaluation


In [ ]:
# 1. GPU Check & Environment Verification
!nvidia-smi
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")


In [ ]:
# 2. Clone Public Repository & Pull Git LFS Weights
!apt-get install -y git-lfs
!git lfs install
!rm -rf CanItDoom
!git clone https://github.com/imagine1phoenix/CanItDoom.git
%cd CanItDoom
!git lfs pull
!ls -lh submission/weights/model.pt


In [ ]:
# 3. Install Dependencies
!pip install --upgrade pip
!pip install uv
!uv pip install --system -e env/


In [ ]:
# 4. Mount Google Drive for Resilient Checkpoint Mirroring
from google.colab import drive
import os
drive.mount("/content/drive")
backup_dir = "/content/drive/MyDrive/CanItDoom_Checkpoints"
os.makedirs(backup_dir, exist_ok=True)
print("Drive checkpoint backup directory ready:", backup_dir)


In [ ]:
# 5. Pre-Training Rule & Baseline Validation Check
!python3 tools/check_rules.py
!python validate.py


In [ ]:
# 6. Launch Recurrent PPO Curriculum Training (Stages 1-5)
# Saves local checkpoints every 15 min and mirrors to Google Drive
!python train/run.py --stages 1,2,3,4,5 --steps-per-stage 1000000 --envs 8 --device cuda --save-interval-sec 900 --drive-backup /content/drive/MyDrive/CanItDoom_Checkpoints


In [ ]:
# 7. Evaluate Checkpoint on 6 Held-Out Evaluation Maps
!python train/evaluate.py --checkpoint checkpoints/ckpt_latest.pt --output eval_results.csv --device cuda
!cat eval_results.csv


In [ ]:
# 8. Synchronize Best Checkpoint to submission/weights/model.pt & Final Verification
!cp checkpoints/ckpt_latest.pt submission/weights/model.pt
!python3 tools/check_rules.py
!python validate.py
